### Why one-hot encoding for `source`

We evaluated the predictive contribution of source information using three controlled setups on the development split only.

Using **source identity with one-hot encoding**, the model achieves a **Macro-F1 of 0.396**, with an overall **accuracy of 0.454** and **macro-averaged recall of 0.395**. Several classes show substantial recall driven purely by editorial bias (e.g., class 0 recall 0.624, class 2 recall 0.598, class 5 recall 0.512), indicating that the source alone provides a non-trivial but coarse discriminative signal.

When replacing source identity with **numerical features derived from source statistics** (support, entropy, and dominant class prior), performance drops sharply to a **Macro-F1 of 0.225** and **accuracy of 0.314**, with **macro recall decreasing to 0.277**. This confirms that aggregating source information into numerical summaries removes class-specific structure and introduces noise.

Finally, combining **one-hot encoded source with source-derived features** yields **no measurable improvement**, with **Macro-F1 = 0.395** and **accuracy = 0.453**, effectively identical to the source-only baseline. This shows that source-derived features are redundant once the source identity is modeled explicitly.

Overall, these results demonstrate that **source information is best captured as a categorical identity via one-hot encoding**, while numerical features derived from source statistics do not add independent predictive power and may degrade performance.


In [7]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score


In [3]:
# Load data and sanity checK
DEV_PATH = "../data/raw/development.csv"
df = pd.read_csv(DEV_PATH)

for col in ["article", "title", "source"]:
	df[col] = df[col].fillna("").astype(str)

label_col = "label"

In [4]:
# Source-Derivate Feature 


# source_support
source_support = df["source"].value_counts()
df["source_support"] = df["source"].map(source_support)

# source_entropy
def entropy(p):
	p = p[p > 0]
	return -(p * np.log2(p)).sum()

src_label_dist = df.groupby("source")[label_col].value_counts(normalize=True)

source_entropy = {}
for src in src_label_dist.index.get_level_values(0).unique():
	source_entropy[src] = entropy(src_label_dist.loc[src].values)

df["source_entropy"] = df["source"].map(source_entropy)

# source_max_prior
source_max_prior = (
	df.groupby("source")[label_col]
	.value_counts(normalize=True)
	.groupby(level=0)
	.max()
)

df["source_max_prior"] = df["source"].map(source_max_prior)

derived_features = [
	"source_support",
	"source_entropy",
	"source_max_prior"
]



In [5]:
# Train /test split on dev (1 fold, stratified)
X = df[["source"] + derived_features]
y = df[label_col]

X_tr, X_te, y_tr, y_te = train_test_split(
	X,
	y,
	test_size=0.2,
	random_state=42,
	stratify=y
)


In [8]:
# Model A - Source identy only (OHE)

model_ohe = Pipeline([
	("pre", ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"])
		],
		remainder="drop"
	)),
	("clf", LogisticRegression(
		max_iter=2000,
		n_jobs=-1,
		class_weight="balanced"
	))
])

model_ohe.fit(X_tr[["source"]], y_tr)
pred_ohe = model_ohe.predict(X_te[["source"]])

print("\n===== SOURCE ONLY | ONE-HOT =====\n")
print("Macro F1:", f1_score(y_te, pred_ohe, average="macro"))
print(classification_report(y_te, pred_ohe, digits=3))



===== SOURCE ONLY | ONE-HOT =====

Macro F1: 0.3957835829622319
              precision    recall  f1-score   support

           0      0.566     0.624     0.593      4709
           1      0.655     0.256     0.368      2118
           2      0.821     0.598     0.692      2232
           3      0.294     0.164     0.211      1995
           4      0.322     0.366     0.342      1715
           5      0.388     0.512     0.441      2611
           6      0.082     0.244     0.123       620

    accuracy                          0.454     16000
   macro avg      0.447     0.395     0.396     16000
weighted avg      0.505     0.454     0.460     16000



In [9]:
model_derived = Pipeline([
	("scaler", StandardScaler()),
	("clf", LogisticRegression(
		max_iter=2000,
		n_jobs=-1,
		class_weight="balanced"
	))
])

model_derived.fit(X_tr[derived_features], y_tr)
pred_derived = model_derived.predict(X_te[derived_features])

print("\n===== SOURCE-DERIVED FEATURES ONLY =====\n")
print(classification_report(y_te, pred_derived, digits=3))
print("Macro F1:", f1_score(y_te, pred_derived, average="macro"))



===== SOURCE-DERIVED FEATURES ONLY =====

              precision    recall  f1-score   support

           0      0.538     0.548     0.543      4709
           1      0.152     0.008     0.015      2118
           2      0.413     0.580     0.483      2232
           3      0.450     0.005     0.009      1995
           4      0.201     0.297     0.240      1715
           5      0.234     0.155     0.186      2611
           6      0.059     0.350     0.101       620

    accuracy                          0.314     16000
   macro avg      0.292     0.277     0.225     16000
weighted avg      0.354     0.314     0.290     16000

Macro F1: 0.2252659178802337


In [10]:
model_combo = Pipeline([
	("pre", ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("num", StandardScaler(), derived_features)
		],
		remainder="drop"
	)),
	("clf", LogisticRegression(
		max_iter=2000,
		n_jobs=-1,
		class_weight="balanced"
	))
])

model_combo.fit(X_tr, y_tr)
pred_combo = model_combo.predict(X_te)

print("\n===== SOURCE OHE + DERIVED FEATURES =====\n")
print(classification_report(y_te, pred_combo, digits=3))
print("Macro F1:", f1_score(y_te, pred_combo, average="macro"))



===== SOURCE OHE + DERIVED FEATURES =====

              precision    recall  f1-score   support

           0      0.566     0.624     0.593      4709
           1      0.678     0.248     0.363      2118
           2      0.824     0.598     0.693      2232
           3      0.294     0.167     0.213      1995
           4      0.319     0.368     0.341      1715
           5      0.387     0.512     0.441      2611
           6      0.082     0.244     0.123       620

    accuracy                          0.453     16000
   macro avg      0.450     0.394     0.395     16000
weighted avg      0.508     0.453     0.459     16000

Macro F1: 0.3954083826218177
